# N139 · 有序集合、Treap与跳表原理

**目标：** 明确有序动态操作需求并实现带秩平衡树。

**先修：** N073, N057, N134, N131。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 旋转；AVL/红黑约束导览；Treap；重复键计数；rank/kth；跳表层级。

## 从问题到算法

普通BST可能退化为链。Treap同时维护按key的BST顺序和随机priority的堆顺序，插入后用旋转恢复堆性质；子树size让rank与kth成为沿树下降的查询。重复值用count保存，不创建相同key节点。AVL/红黑树提供最坏高度保证，Treap与跳表则依赖随机化的期望分析。

## 最小实现

**本章接口：** `OrderStatisticTreap`、`contains_nearby_almost_duplicate(nums, index_diff, value_diff)`

In [1]:
from random import Random

class _TreapNode:
    def __init__(self,key,priority):
        self.key=key; self.priority=priority; self.count=1; self.size=1; self.left=self.right=None

def _size(node): return node.size if node else 0

def _pull(node):
    if node: node.size=node.count+_size(node.left)+_size(node.right)
    return node

def _rotate_left(node):
    root=node.right; node.right=root.left; root.left=node; _pull(node); return _pull(root)

def _rotate_right(node):
    root=node.left; node.left=root.right; root.right=node; _pull(node); return _pull(root)

class OrderStatisticTreap:
    def __init__(self,rng=None): self.root=None; self.rng=Random(139) if rng is None else rng; self.serial=0
    def __len__(self): return _size(self.root)
    def __contains__(self,key):
        node=self.root
        while node:
            if node.key==key: return True
            node=node.left if key<node.key else node.right
        return False
    def insert(self,key):
        self.serial+=1; priority=(self.rng.getrandbits(64),self.serial)
        def put(node):
            if node is None: return _TreapNode(key,priority)
            if key==node.key: node.count+=1
            elif key<node.key:
                node.left=put(node.left)
                if node.left.priority<node.priority: node=_rotate_right(node)
            else:
                node.right=put(node.right)
                if node.right.priority<node.priority: node=_rotate_left(node)
            return _pull(node)
        self.root=put(self.root)
    def _merge(self,left,right):
        if not left: return right
        if not right: return left
        if left.priority<right.priority: left.right=self._merge(left.right,right); return _pull(left)
        right.left=self._merge(left,right.left); return _pull(right)
    def discard(self,key):
        if key not in self: return False
        def erase(node):
            if key<node.key: node.left=erase(node.left)
            elif key>node.key: node.right=erase(node.right)
            elif node.count>1: node.count-=1
            else: return self._merge(node.left,node.right)
            return _pull(node)
        self.root=erase(self.root); return True
    def rank(self,key):
        node=self.root; answer=0
        while node:
            if key<=node.key: node=node.left
            else: answer+=_size(node.left)+node.count; node=node.right
        return answer
    def kth(self,k):
        if not 0<=k<len(self): raise IndexError('zero-based rank outside multiset')
        node=self.root
        while node:
            left=_size(node.left)
            if k<left: node=node.left
            elif k<left+node.count: return node.key
            else: k-=left+node.count; node=node.right
    def lower_bound(self,key):
        node=self.root; answer=None
        while node:
            if node.key>=key: answer=node.key; node=node.left
            else: node=node.right
        return answer

def contains_nearby_almost_duplicate(nums,index_diff,value_diff):
    if index_diff<=0 or value_diff<0: return False
    tree=OrderStatisticTreap()
    for i,x in enumerate(nums):
        candidate=tree.lower_bound(x-value_diff)
        if candidate is not None and candidate<=x+value_diff: return True
        tree.insert(x)
        if i>=index_diff: tree.discard(nums[i-index_diff])
    return False

## 正确性、前提与复杂度

旋转不改变中序顺序，只改变父子关系，因此BST不变量保持；按优先级旋转使随机优先级最小的点成为子树根。rank累加已确定小于目标的左子树与重复数；kth根据左子树大小排除不含答案的一侧。滑窗树只保存最近index_diff个先前元素，再查询[value−t,value+t]。

**代价：** 独立随机优先级下树高与操作期望O(log n)，最坏仍可O(n)；递归实现也受最坏树高限制。空间O(不同key数)。固定随机源仅保证测试复现，不把期望保证写成最坏保证；跳表/AVL/红黑树只作结构对照，未伪装为本章已实现。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

tree=OrderStatisticTreap(Random(139))
for x in [5,2,8,2,6,1]: tree.insert(x)
rows=[]; stack=[tree.root]
while stack:
    node=stack.pop()
    if not node: continue
    rows.append((node.key,node.count,node.size,node.left.key if node.left else None,node.right.key if node.right else None))
    stack.extend([node.right,node.left])
show_table(['key','重复次数','子树size','左孩子','右孩子'],rows)
print('全部顺序统计',[tree.kth(i) for i in range(len(tree))])

key,重复次数,子树size,左孩子,右孩子
1,1,6,None,8
8,1,5,2,None
2,2,4,None,6
6,1,2,5,None
5,1,1,None,None


全部顺序统计 [1, 2, 2, 5, 6, 8]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from bisect import bisect_left,insort
rng=Random(139); tree=OrderStatisticTreap(Random(5)); reference=[]
def audit(node,low=None,high=None):
    if not node: return 0
    assert (low is None or low<node.key) and (high is None or node.key<high)
    for child in (node.left,node.right):
        if child: assert node.priority<child.priority
    total=node.count+audit(node.left,low,node.key)+audit(node.right,node.key,high)
    assert node.count>=1 and node.size==total
    return total
for _ in range(700):
    x=rng.randrange(-12,13)
    if rng.randrange(2): tree.insert(x); insort(reference,x)
    else:
        found=x in reference; assert tree.discard(x)==found
        if found: reference.remove(x)
    assert audit(tree.root)==len(reference)==len(tree)
    assert [tree.kth(i) for i in range(len(tree))]==reference
    assert tree.rank(x)==bisect_left(reference,x)
    candidates=[y for y in reference if y>=x]
    assert tree.lower_bound(x)==(candidates[0] if candidates else None)
for _ in range(180):
    a=[rng.randrange(-20,21) for _ in range(12)]; k=rng.randrange(6); t=rng.randrange(-1,8)
    want=any(j-i<=k and abs(a[j]-a[i])<=t for i in range(len(a)) for j in range(i+1,len(a)))
    assert contains_nearby_almost_duplicate(a,k,t)==want
print('N139: 所有本章断言通过')

N139: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 说明随机平衡是期望界而非每次保证。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较bisect加列表插入的成本。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 220. Contains Duplicate III（canonical）
- 1206. Design Skiplist（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。